User Input (Product + Audience + Platform + Tone)
        ↓
1. Campaign Strategy Generator (LLM)
        ↓
2. Ad Script Generator (LLM)
        ↓
3. Storyboard Prompt Generator (LLM)
        ↓
4. Video Generation Layer
   → Option A: OpenAI Sora API
   → Option B: Stable Video Diffusion (HF)
   → Option C: ModelScope Text2Video
        ↓
5. Voiceover Generation (TTS)
        ↓
6. Video Composer (MoviePy)
        ↓
Final Branded Ad Output


In [1]:
!pip install transformers accelerate torch diffusers moviepy imageio imageio-ffmpeg TTS


ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0, <3.9; 0.0.10.3 Requires-Python >=3.6.0, <3.9; 0.0.11 Requires-Python >=3.6.0, <3.9; 0.0.12 Requires-Python >=3.6.0, <3.9; 0.0.13.1 Requires-Python >=3.6.0, <3.9; 0.0.13.2 Requires-Python >=3.6.0, <3.9; 0.0.14.1 Requires-Python >=3.6.0, <3.9; 0.0.15 Requires-Python >=3.6.0, <3.9; 0.0.15.1 Requires-Python >=3.6.0, <3.9; 0.0.9 Requires-Python >=3.6.0, <3.9; 0.0.9.1 Requires-Python >=3.6.0, <3.9; 0.0.9.2 Requires-Python >=3.6.0, <3.9; 0.0.9a10 Requires-Python >=3.6.0, <3.9; 0.0.9a9 Requires-Python >=3.6.0, <3.9; 0.1.0 Requires-Python >=3.6.0, <3.10; 0.1.1 Requires-Python >=3.6.0, <3.10; 0.1.2 Requires-Python >=3.6.0, <3.10; 0.1.3 Requires-Python >=3.6.0, <3.10; 0.10.0 Requires-Python >=3.7.0, <3.11; 0.10.1 Requires-Python >=3.7.0, <3.11; 0.10.2 Requires-Python >=3.7.0, <3.11; 0.11.0 Requires-Python >=3.7.0, <3.11; 0.11.1 Requires-Python >=3.7.0, <3.11; 0.12.0 Requires-Python >=3

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

def generate_ad_script(product, audience, platform, tone):
    prompt = f"""
    You are a professional ad campaign strategist.

    Create:
    1. A short high-converting video ad script (30 seconds)
    2. Scene-by-scene storyboard prompts for AI video generation

    Product: {product}
    Target Audience: {audience}
    Platform: {platform}
    Tone: {tone}

    Output format:
    SCRIPT:
    ...

    STORYBOARD:
    Scene 1:
    Scene 2:
    ...
    """

    output = generator(
        prompt,
        max_new_tokens=600,
        temperature=0.7,
        do_sample=True
    )

    return output[0]["generated_text"]

# Example
script_output = generate_ad_script(
    product="AI-powered HR automation platform",
    audience="HR Managers in mid-size companies",
    platform="LinkedIn",
    tone="Professional and innovative"
)

print(script_output)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



    You are a professional ad campaign strategist.

    Create:
    1. A short high-converting video ad script (30 seconds)
    2. Scene-by-scene storyboard prompts for AI video generation

    Product: AI-powered HR automation platform
    Target Audience: HR Managers in mid-size companies
    Platform: LinkedIn
    Tone: Professional and innovative

    Output format:
    SCRIPT:
    ...
    
    STORYBOARD:
    Scene 1:
    Scene 2:
    ...
    
    SCRIPT:

    (Scene 1: HR Manager sitting at a desk, looking overwhelmed with paperwork)
    Narrator (Voiceover): "As an HR manager in a mid-size company, you wear many hats."

    HR Manager: "I'm constantly juggling recruitment, onboarding, and employee management – it's never-ending."

    (Scene 2: AI logo appears on the screen, followed by a shot of the HR Manager staring at her computer)

    Narrator (Voiceover): "But what if there was a way to streamline your HR processes?"

    (Scene 3: Montage of HR tasks: posting job ads, s

In [3]:
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16
)
pipe.to("cuda")

video_frames = pipe(
    "Corporate team using AI dashboard in modern office, cinematic lighting"
).frames


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--damo-vilab--text-to-video-ms-1.7b/snapshots/8227dddca75a8561bf858d604cc5dae52b954d01/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


OutOfMemoryError: CUDA out of memory. Tried to allocate 58.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 39.81 MiB is free. Including non-PyTorch memory, this process has 14.52 GiB memory in use. Of the allocated memory 14.32 GiB is allocated by PyTorch, and 71.34 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from TTS.api import TTS

tts = TTS(model_name="tts_models/en/ljspeech/tacotron2-DDC")

def generate_voiceover(text, output_file="voiceover.wav"):
    tts.tts_to_file(text=text, file_path=output_file)


In [ ]:
from moviepy.editor import ImageSequenceClip, AudioFileClip

def create_final_video(frames, audio_path, output_path="final_ad.mp4"):
    clip = ImageSequenceClip(frames, fps=24)
    audio = AudioFileClip(audio_path)

    final_clip = clip.set_audio(audio)
    final_clip.write_videofile(output_path, codec="libx264")


In [ ]:
def platform_formatting(platform):
    if platform == "Instagram":
        return {"ratio": "9:16", "duration": 15}
    elif platform == "YouTube":
        return {"ratio": "16:9", "duration": 30}
    elif platform == "LinkedIn":
        return {"ratio": "1:1", "duration": 30}


In [ ]:
def run_campaign(product, audience, platform, tone):

    # 1. Generate Script + Storyboard
    campaign = generate_ad_script(product, audience, platform, tone)

    # 2. Extract storyboard prompt (simplified)
    video_prompt = campaign.split("STORYBOARD:")[-1]

    # 3. Generate Video
    frames = generate_video_from_prompt(video_prompt)

    # 4. Generate Voiceover
    generate_voiceover(campaign)

    # 5. Compose Final Video
    create_final_video(frames, "voiceover.wav")

    print("Ad Campaign Video Created Successfully!")
